# sessions_conditions.ipynb
* пути к файлам с данными для подключения к SQL серверу;
* загрузка данных SQL таблиц для дальнейшей работы;
* Транспонирование методом  [ pivot_table ];
* Преобразование времени из минут в формат ЧЧ:ММ;
* Вызываем функцию слияния ДФ с проверкой наличия колонок и размерности;
* Удаляем колонки, начинающиеся с "1_"; в последствии вернём когда настроим окна;
* Удаляем префикс "0_" из имен колонок;

In [1]:
# Динамический импорт и инициализация библиотек и путей для работы с данными в Python <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
from pathlib import Path
import csv
import sys
import os
import pandas as pd
import json
from collections import defaultdict

sub_project_dir = "trading_conditions"

file_dir = os.getcwd()                                                              # Определяем путь к текущему файлу (где выполняется код)
print(f"Файл в директории:                                      {file_dir}")

project_dir                     = Path(file_dir).parent.parent                      # Переход на уровень выше (fc_to_mt5_migrations/own_platform)
print(f"Рабочая директория проекта:                             {project_dir}")

parent_dir                      = Path.cwd().parent.parent.parent                   # Переход на уровень выше (fc_to_mt5_migrations)
print(f"Рабочая директория проекта для доступа к библиотекам:   {parent_dir}")

directory_data_log_files        = os.path.join(project_dir, sub_project_dir, 'log_data_files')
print(f"[directory_data_log_files];     Путь к каталогу с лог-файлами:                          {directory_data_log_files}")

directory_data_temp_files       = os.path.join(project_dir, sub_project_dir, 'working_data_files') 
print(f"[directory_data_temp_files];    Путь к каталогу с временными файлами:                   {directory_data_temp_files}")

directory_data_original_data    = os.path.join(project_dir, 'original_data')        # Путь к каталогу с оригинальными данными
print(f"Путь к каталогу с оригинальными данными:                {directory_data_original_data}")

directory_data_set              = os.path.join(project_dir, 'data_set')             # Путь к каталогу с конфигурационными данными
print(f"Путь к каталогу с файлами настроек:                     {directory_data_set}")

directory_data_output           = os.path.join(parent_dir, 'output_data')           # Путь к каталогу с выходными данными

libraries_path = os.path.join(parent_dir, "libraries_py")                           # Формируем путь к libraries_py каталогу с библиотеками *.py
sys.path.append(libraries_path)                                                     # sys.path — это список путей, где Python ищет модули при import module_name.

if libraries_path in sys.path: print(f"✅ Каталог {libraries_path} успешно добавлен в sys.path")
else: print(f"❌ Ошибка: {libraries_path} не найден в sys.path")

# Динамически импорт необходимых функций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
file_imports = "dynamic_import_functions.py"                                        # Библиотека для динамического импорта
file_imports_path = os.path.join(libraries_path, file_imports)
if os.path.exists(file_imports_path):
    import importlib
    importlib.invalidate_caches()                                                   # Сбрасываем кэш перед импортом
    from dynamic_import_functions import import_functions, print_import_function_info
    print(f"\n ✅ Импорт [{file_imports}] успешен.")
else:
    print(f"\n ERROR: Файл '{file_imports}' не найден по пути {file_imports_path}, импорт не выполнен.\n")

modules_to_import = {                                                               # Формируем словарь, с именами файлов и функциями в них
    "yar_sed_general_lib":
        [libraries_path,
                "pd_set_option",                     # Вывод ДФ
                "df_to_csv",                         # Сохранение ДФ в CSV 
                #"CSVLoader",
                "save_data_log_work_file",
                #"detect_encoding",
                "time_to_minutes",
                #"load_string_list",
                "list_print",
                "move_column",
                #"filter_df_by_suffix",
                "check_columns_exist_id_df",
                "merge_left_with_check"],           # Проверка наличия колонок в DataFrame 
    "sql_request_2":
        [libraries_path, 
                "pd_read_sql",
                "get_sql_tab"]
                }

imported = import_functions(modules_to_import)          # Импортируем модули из словаря modules_to_import

print_import_function_info(modules_to_import, imported) # Выводим переменные ожидаемые импортированными функциями 

Файл в директории:                                      c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\ipynb_files
Рабочая директория проекта:                             c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform
Рабочая директория проекта для доступа к библиотекам:   c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations
[directory_data_log_files];     Путь к каталогу с лог-файлами:                          c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\log_data_files
[directory_data_temp_files];    Путь к каталогу с временными файлами:                   c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\working_data_files
Путь к каталогу с оригинальными данными:                c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\original_data
Путь к каталогу с файлами настроек:                 

In [2]:
# [ФУНКЦИЯ] Сортировка колонок в нужном порядке <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
def sort_session_columns(cols):                                         # Функция для сортировки колонок в нужном порядке
    print("[внутренняя ФУНКЦИЯ] Сортировка колонок в нужном порядке.")
    def sort_key(col):
        if col == 'symbolId':   return (-1, '', '', '')                 # всегда первая
        try:                                                            # Разбиваем имя колонки: <type>_<field>_<day>
            parts = col.split('_')
            stype = parts[0]        # trade / quote
            field = parts[1]        # open / close
            day = int(parts[2])     # номер дня
        except: stype, field, day = '', '', 0           # для колонок, которые не подходят под шаблон
        field_order = 0 if field=='open' else 1         # Ключ сортировки: день, open/close, trade/quote
        stype_order = 0 if stype=='trade' else 1        # Ключ сортировки: день, open/close, trade/quote
        return (day, field_order, stype_order)
    return sorted(cols, key=sort_key)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> [ФУНКЦИЯ] Сортировка колонок в нужном порядке

# Функция для проверки словарей на одинаковые ключи, значения и пары key:value <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
def check_dicts(dicts_with_names):  

    key_to_dicts = defaultdict(list)          # key -> [dict_name]
    value_to_entries = defaultdict(list)      # value -> [(dict_name, key)]
    kv_to_dicts = defaultdict(list)            # (key, value) -> [dict_name]

    for dict_name, d in dicts_with_names:                       # Сбор информации
        for k, v in d.items():
            key_to_dicts[k].append(dict_name)
            value_to_entries[v].append((dict_name, k))
            kv_to_dicts[(k, v)].append(dict_name)

    same_keys = {                                                   # 1. Одинаковые ключи
        k: names
        for k, names in key_to_dicts.items()
        if len(names) > 1
    }

    same_values = {                                                 # 2. Одинаковые значения
        v: entries
        for v, entries in value_to_entries.items()
        if len(entries) > 1
    }

    same_key_values = {                                             # 3. Одинаковые пары key:value
        (k, v): names
        for (k, v), names in kv_to_dicts.items()
        if len(names) > 1
    }

    return {"same_keys": same_keys,"same_values": same_values,"same_key_values": same_key_values,}
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> [ФУНКЦИЯ] Проверка словарей на одинаковые ключи, значения и пары key:value

def filter_by_suffix(df, suffix_list):
    print(f" \n \n Создаём колонку с  сответствующими суффиксами из списка:")
    imported["list_print"](suffix_list, "suffix_list")
    df = symbols_df[symbols_df["name"].str.endswith(tuple(suffix_list))].copy()
    df["suffix"] = df["name"].str.extract(r'(\..*)')
    unique_df = df[["marketId", "suffix"]].drop_duplicates().copy()

    cols_map = {"name": "name_market"}   
    unique_df, not_found, not_used = imported['merge_left_with_check']( # Вызываем функцию слияния ДФ с проверкой наличия колонок и размерности
    df_left= unique_df, df_right= symbols_markets_df, left_on= "marketId", right_on= "id",  cols_map=cols_map)

    #imported["pd_set_option"]("unique_df", unique_df, 50)
    return unique_df


def create_market_dict(df, key_col='name_market', value_col='id'):          # Создаёт словарь {name_market: id} с проверкой на один ключ — несколько значений.
    unique_counts = df.groupby(key_col)[value_col].nunique()                # Группируем по ключу и считаем количество уникальных значений
    multi_value_keys = unique_counts[unique_counts > 1].index.tolist()      # Ключи с несколькими уникальными значениями
    
    if multi_value_keys:
        print(f"Внимание: Один ключ соответствует нескольким разным значениям для {len(multi_value_keys)} ключ(ей): {multi_value_keys}")

        for key in multi_value_keys:                                        # Детали для каждого проблемного ключа
            values = df[df[key_col] == key][value_col].unique().tolist()
            print(f"  Ключ '{key}' → значения: {values}")
        
        df_dedup = df.drop_duplicates(subset=key_col, keep='first')         # Обработка: берём первое значение (или можно бросить ошибку/выбрать вручную)
        print("  Для словаря взяты первые значения для каждого ключа.")
    else:
        print("Каждому ключу соответствует ровно одно значение.")
        df_dedup = df.copy()
    
    market_dict = dict(zip(df_dedup[key_col], df_dedup[value_col]))         # Создаём словарь
    
    return market_dict

# Сохраняем в CSV Новое расписание по методу обработки отдельно каждого символа <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
def process_sessions_conditions(table_name, dfs, symbols_df, directory_data_temp_files, directory_data_log_files):

    def time_to_minutes(t):                         # Функция для конвертации времени в минуты
        if t is None:
            return None
        hours, minutes = map(int, t.split(":"))
        return hours * 60 + minutes

    def duplicate_sessions_with_type(df):
        df['type'] = 1
        df_duplicates = df.copy()                                       # 1. Создаём копию для дубликатов
        df_duplicates['type'] = 0                                       # меняем type на 0
        df_combined = pd.concat([df, df_duplicates], ignore_index=True) # 2. Конкатенируем оригинал и дубликаты
        df_combined['_dup'] = [0,1]* (len(df))# 3. Сортируем так, чтобы дубликат шёл сразу после оригинала # создаём вспомогательный индекс: оригинал 0, дубликат 1
        df_combined = df_combined.sort_values(['symbolId','day','_dup']).drop(columns='_dup').reset_index(drop=True)
        df = df_combined        # 4. Обновляем исходный df
        return df

    cols_map = {"name":"name_s"}
    df, symbols_not_found_in_symbols_df, symbols_not_used_in_sessions = imported['merge_left_with_check'](
        df_left= dfs, df_right= symbols_df, left_on= "symbolId", right_on= "symbolId", cols_map=cols_map)

    for col in ["open", "close"]: df[col] = df[col].apply(time_to_minutes)  # Применяем ко всем колонкам open и close перевод времени в минуты с начала суток
    df = duplicate_sessions_with_type(df)
    imported["save_data_log_work_file"](df, table_name+"_sessions.csv", directory_data_temp_files, directory_data_log_files)

    return df


In [4]:
# пути к файлам с данными для подключения к SQL серверу <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
#dot_big_sql_main_cred= "own_platform\\credits\\own_platforn_sql_main_main_01.txt"           # Получаем Данные с ПРОД сервера
dot_big_sql_main_cred= "own_platform\\credits\\own_platforn_sql_main_stage_01.txt"
print(f"Путь к файлу с данными для подключения к SQL серверу: [ {dot_big_sql_main_cred} ]")

# загрузка данных SQL таблиц для дальнейшей работы <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
symbols_sessions_df = imported["get_sql_tab"]   ("SELECT * FROM `symbolsSessions`",    dot_big_sql_main_cred) 
symbols_df          = imported["get_sql_tab"]   ("SELECT * FROM `symbols`",            dot_big_sql_main_cred)
symbols_markets_df  = imported["get_sql_tab"]   ("SELECT * FROM `symbolsMarkets`",     dot_big_sql_main_cred)

Путь к файлу с данными для подключения к SQL серверу: [ own_platform\credits\own_platforn_sql_main_stage_01.txt ]
получаем данные SELECT * FROM `symbolsSessions`; из: 10.1.0.8 pma.y.d main_stage_01
получаем данные SELECT * FROM `symbols`; из: 10.1.0.8 pma.y.d main_stage_01
получаем данные SELECT * FROM `symbolsMarkets`; из: 10.1.0.8 pma.y.d main_stage_01


In [ ]:
# [ НЕ обязателен ] Выводим загруженные данные для проверки <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''

symbols_sessions_df = symbols_sessions_df.sort_values   (                                   # последовательная сортировка по нескольким колонкам
                                                        by=["symbolId", "day", "type"],     # список колонок для сортировки
                                                        ascending=[True, True, True],       # направление для каждой колонки
                                                        ignore_index=True                   # сброс индексов после сортировки
                                                        )
imported["pd_set_option"]("symbols_sessions_df", symbols_sessions_df, 5)

df_filtered = symbols_sessions_df[symbols_sessions_df["symbolId"] == 2]
imported["pd_set_option"]("symbols_sessions_df => df_filtered", df_filtered, 50)
#symbols_sessions_df = df_filtered

imported["pd_set_option"]("symbols_df", symbols_df, 5)
df_filtered = symbols_df[symbols_df["marketId"] == 18]
imported["pd_set_option"]("df_filtered", df_filtered, 50)

imported["pd_set_option"]("symbols_markets_df", symbols_markets_df, 100)

In [8]:
df_filtered = symbols_sessions_df[symbols_sessions_df["symbolId"] == 5978]
print("symbols_sessions_df[\"symbolId\"] == 1190: ", len(df_filtered))
imported["pd_set_option"]("df_filtered", df_filtered, 50)

symbols_sessions_df["symbolId"] == 1190:  20

df_filtered  (20 строк × 6 колонок)


,id,symbolId,type,day,open,close
12932,12933,5978,1,1,0,1259
12933,12934,5978,0,1,0,1259
12934,12935,5978,1,1,1320,1439
12935,12936,5978,0,1,1320,1439
12936,12937,5978,1,2,0,1259
12937,12938,5978,0,2,0,1259
12938,12939,5978,1,2,1320,1439
12939,12940,5978,0,2,1320,1439
12940,12941,5978,1,3,0,1259
12941,12942,5978,0,3,0,1259


In [5]:
# Преобразование расписания торговых сессий из формата торгового сервера, в читаемую таблицу <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
"""''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''"""
# Транспонирование методом  [ pivot_table ] <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print("Транспонирование серверного формата данных методом  [ pivot_table ].")
df_sessions_transposed = symbols_sessions_df.pivot_table(index='symbolId',values=['open','close'],columns=['type','day'],aggfunc='first')
df_sessions_transposed.columns = [f"{stype}_{field}_{day}" for field, stype, day in df_sessions_transposed.columns] # Flatten MultiIndex в имена колонок
df_sessions_transposed = df_sessions_transposed.reset_index()

ordered_cols = sort_session_columns(df_sessions_transposed.columns) # [ ФУНКЦИЯ ] Вызываем функцию сортировки колонок
df_sessions_transposed = df_sessions_transposed[ordered_cols]
#imported["pd_set_option"]("[df_sessions_transposed] время в минутах с начала суток", df_sessions_transposed, 5) 

# Преобразование времени из минут в формат ЧЧ:ММ <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print("Преобразование времени из минут в формат ЧЧ:ММ.")
df = df_sessions_transposed.copy()
time_cols = df.filter(regex='_open_|_close_').columns                                                           # Все колонки с _open_ или _close_
df[time_cols] = df[time_cols].applymap(lambda x: pd.NA if pd.isna(x) else f"{int(x)//60:02d}:{int(x)%60:02d}")
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> Преобразование времени из минут в формат ЧЧ:ММ

df_sessions_transposed = df.copy()
#imported["pd_set_option"]("[df_sessions_transposed] время в читаемом виде", df_sessions_transposed, 5)

# Создаём агрегированную таблицу с торговыми сессиями включающую РЫНКИ и БИРЖЕВЫЕ тикеры торговых инструметов <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< 
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
cols_map = {                                                                            # [ФУНКЦИЯ] слияния ДФ с проверкой наличия колонок и размерности 
    "name":"name_s", "displayName":"displayName_s", "marketId":"marketId_s", "tradeMode":"tradeMode_s"}
df_sessions_enriched, symbols_not_found_in_symbols_df, symbols_not_used_in_sessions = imported['merge_left_with_check'](
    df_left= df_sessions_transposed, df_right= symbols_df, left_on= "symbolId", right_on= "symbolId", cols_map=cols_map)

imported["list_print"](symbols_not_found_in_symbols_df, 'symbols_not_found_in_symbols_df')
imported["list_print"](set(symbols_not_used_in_sessions), 'set(symbols_not_used_in_sessions)')

cols_map = {                                                                            # [ФУНКЦИЯ] слияния ДФ с проверкой наличия колонок и размерности
    "name": "name_market"}   
df_sessions_enriched, not_found, not_used = imported['merge_left_with_check']( # Вызываем функцию слияния ДФ с проверкой наличия колонок и размерности
    df_left= df_sessions_enriched, df_right= symbols_markets_df, left_on= "marketId_s", right_on= "id",  cols_map=cols_map)

# [ФУНКЦИЯ] перемещения колонок <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
df = df_sessions_enriched.copy()
list_col_name = ['name_s', 'displayName_s', 'marketId_s', 'name_market','tradeMode_s', 'id']
add_list_col_name = False
new_position = 0
new_position_step = 1
df_sessions_enriched = imported["move_column"](df, list_col_name, add_list_col_name, new_position, new_position_step).copy()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

print('\n',"Сохранение итоговой таблицы с торговыми сессиями в CSV файл, для использования в [visualization_trading_sessions.ipynb]")
imported["save_data_log_work_file"](df_sessions_enriched, "df_sessions_enriched.csv", directory_data_temp_files, directory_data_log_files)
imported["pd_set_option"]("[df_sessions_enriched] агрегированная таблица с торговыми сессиями включающая РЫНКИ и БИРЖЕВЫЕ тикеры торговых инструметов", df_sessions_enriched, 5)

Транспонирование серверного формата данных методом  [ pivot_table ].
[внутренняя ФУНКЦИЯ] Сортировка колонок в нужном порядке.
Преобразование времени из минут в формат ЧЧ:ММ.


C:\Users\nigilist\AppData\Local\Temp\ipykernel_26252\3245044306.py:17: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[time_cols] = df[time_cols].applymap(lambda x: pd.NA if pd.isna(x) else f"{int(x)//60:02d}:{int(x)%60:02d}")


📝 [ 3 ] элементов в списке [ symbols_not_found_in_symbols_df ] список: [10220, 10221, 10222]
📝 [ 2120 ] элементов в списке [ set(symbols_not_used_in_sessions) ] список: {5, 6, 19, 23, 24, 37, 38, 41, 42, 55, 56, 57, 58, 66, 67, 71, 72, 73, 74, 75, 106, 126, 127, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 161, 162, 164, 165, 168, 169, 174, 180, 181, 183, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 198, 199, 201, 202, 204, 205, 206, 207, 208, 209, 210, 211, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 230, 231, 232, 233, 234, 235, 236, 237, 241, 243, 244, 245, 247, 249, 254, 255, 256, 259, 260, 261, 262, 265, 266, 268, 269, 271, 272, 273, 274, 275, 276, 279, 280, 282, 283, 284, 285, 286, 288, 289, 291, 294, 295, 297, 298, 299, 300, 301, 302, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 32

,name_s,displayName_s,marketId_s,name_market,tradeMode_s,id,symbolId,0_open_0,1_open_0,0_close_0,1_close_0,0_open_1,1_open_1,0_close_1,1_close_1,0_open_2,1_open_2,0_close_2,1_close_2,0_open_3,1_open_3,0_close_3,1_close_3,0_open_4,1_open_4,0_close_4,1_close_4,0_open_5,1_open_5,0_close_5,1_close_5,0_open_6,1_open_6,0_close_6,1_close_6
0,AUDCAD,AUD / CAD,2.0,Minor,4.0,2.0,1,21:00,21:00,23:59,23:59,00:00,00:00,23:59,23:59,00:00,00:00,23:59,23:59,00:00,00:00,23:59,23:59,00:00,00:00,23:59,23:59,00:00,00:00,20:59,20:59,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1953,NaN,NaN,NaN,NaN,NaN,NaN,10222,<NA>,<NA>,<NA>,<NA>,00:00,00:00,06:30,06:30,00:00,00:00,06:30,06:30,00:00,00:00,06:30,06:30,00:00,00:00,06:30,06:30,00:00,00:00,06:30,06:30,<NA>,<NA>,<NA>,<NA>


In [6]:
df = df_sessions_enriched.copy()
df_filtered = df[df["name_s"] == "Copper"]
print("symbols_df[\"marketId\"] == 18: ", len(df_filtered))
imported["pd_set_option"]("df_filtered", df_filtered, 5)

symbols_df["marketId"] == 18:  1

df_filtered  (1 строк × 35 колонок)


,name_s,displayName_s,marketId_s,name_market,tradeMode_s,id,symbolId,0_open_0,1_open_0,0_close_0,1_close_0,0_open_1,1_open_1,0_close_1,1_close_1,0_open_2,1_open_2,0_close_2,1_close_2,0_open_3,1_open_3,0_close_3,1_close_3,0_open_4,1_open_4,0_close_4,1_close_4,0_open_5,1_open_5,0_close_5,1_close_5,0_open_6,1_open_6,0_close_6,1_close_6
1190,Copper,Copper,37.0,Spot Commodities,4.0,37.0,5978,<NA>,<NA>,<NA>,<NA>,00:00,00:00,20:59,20:59,00:00,00:00,20:59,20:59,00:00,00:00,20:59,20:59,00:00,00:00,20:59,20:59,00:00,00:00,20:59,20:59,<NA>,<NA>,<NA>,<NA>


    | 
    Подготовка и Сохранение отчёта по текущим расписаниям торговли на сервере
###### Переместить в отдельный который будет использовать таблицу сгенерированную ранее


    >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> Завершение блока отчётов
    |

In [ ]:
# Проверка на наличие МАРКЕТА у инструмента разрешённого к торговле <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
df_filtered = df_sessions_enriched[df_sessions_enriched["name_market"].isna() & (df_sessions_enriched["tradeMode_s"]==4)]
marketId_error_df = df_filtered[["name_s", "displayName_s", "marketId_s", "name_market", "symbolId"]].copy()
imported["save_data_log_work_file"](marketId_error_df, "marketId_error_df.csv", directory_data_temp_files, directory_data_log_files)

imported["pd_set_option"]("df_filtered", marketId_error_df, 50)

In [ ]:
df = df_sessions_enriched.copy()
unique_df = df[["marketId_s", "name_market"]].drop_duplicates()
imported["pd_set_option"]("unique_df", unique_df, 5)

In [ ]:
# Определение списков суффиксов для фильтрации торговых инструментов по РЫНКАМ и БИРЖЕВЫМ тикерам для США и Европы <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# Создание словарей с РЫНКАМИ и БИРЖЕВЫМИ тикерами торговых инструметов для США и Европы <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

BR_suffix_list  = ['.BR',]
CL_suffix_list  = ['.CL',]
US_suffix_list  = ['.ETF','.A','.N','.OQ','.TO','.O']                                                           # Список суффиксов для США
EU_suffix_list  = ['.AS','.S','.L','.DE','.PA','.SE','.IT','.LS','.AT','.FI','.XE','.SAN','.BE','.MAC', 'SA']   # Список суффиксов для Европы

suffix_list = US_suffix_list
USA_markets_df = filter_by_suffix(symbols_df, suffix_list)
USA_market_dict = create_market_dict(USA_markets_df)                                          # Создание словаря {name_market: id} Американские акции

suffix_list = EU_suffix_list
EU_market_df = filter_by_suffix(symbols_df, suffix_list)
EU_market_dict = create_market_dict(EU_market_df)                                             # Создание словаря {name_market: id} Европейские акции

In [ ]:

dicts = [ # Список кортежей (имя_словаря, словарь) для проверки на одинаковые ключи, значения и пары key:value
    ("USA_market_dict", USA_market_dict),
    ("EU_market_dict", EU_market_dict)]

def check_dicts(dicts_with_names):  # Функция для проверки словарей на одинаковые ключи, значения и пары key:value

    key_to_dicts = defaultdict(list)          # key -> [dict_name]
    value_to_entries = defaultdict(list)      # value -> [(dict_name, key)]
    kv_to_dicts = defaultdict(list)            # (key, value) -> [dict_name]

    for dict_name, d in dicts_with_names:                       # Сбор информации
        for k, v in d.items():
            key_to_dicts[k].append(dict_name)
            value_to_entries[v].append((dict_name, k))
            kv_to_dicts[(k, v)].append(dict_name)

    same_keys = {                                                   # 1. Одинаковые ключи
        k: names
        for k, names in key_to_dicts.items()
        if len(names) > 1
    }

    same_values = {                                                 # 2. Одинаковые значения
        v: entries
        for v, entries in value_to_entries.items()
        if len(entries) > 1
    }

    same_key_values = {                                             # 3. Одинаковые пары key:value
        (k, v): names
        for (k, v), names in kv_to_dicts.items()
        if len(names) > 1
    }

    return {"same_keys": same_keys,"same_values": same_values,"same_key_values": same_key_values,}

result = check_dicts(dicts)# Функция для проверки словарей на одинаковые ключи, значения и пары key:value

print("1) Одинаковые ключи:")
for k, dicts in result["same_keys"].items(): print(f"  ключ '{k}' → {dicts}")

print("2) Одинаковые значения:")
for v, entries in result["same_values"].items():
    print(f"  значение '{v}':")
    for dict_name, key in entries:
        print(f"    {dict_name}[{key}]")

print("3) Одинаковые key:value:")
for (k, v), dicts in result["same_key_values"].items(): print(f"  '{k}: {v}' → {dicts}")


In [ ]:
# Кластера торговых, сессий Биржевых индексов <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
index_sessions_dict = {
                        'asia_oceania_index_Cluster':
                            {          
                                ('AUS200','N225','HSI','HKD50',):                       # внутрисуточный разрыв 22:00 – 22:59 UTC
                                                {(0,):{"o":"23:00", "c":"23:59"}, (1, 2, 3, 4,):{"o":"00:00", "c":"23:59"}, (5,):{"o":"00:00", "c":"21:59"},(6,):{"o":None, "c":None}},
                                ('TOPIX','KOSPI',):
                                                {(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,)   :{"o":"00:00", "c":"06:30"},(6,):{"o":None, "c":None}},
                                ('CHINA50',)    :{(0,):{"o":None, "c":None},(1, 2, 3, 4,)     :{"o":"01:00", "c":"20:59"}, (5,):{"o":"01:00", "c":"20:59"},(6,):{"o":None, "c":None},},
                                ('SG25',)       :{(0,):{"o":None, "c":None},(1, 2, 3, 4,)     :{"o":"00:30", "c":"20:59"}, (5,):{"o":"00:30", "c":"20:59"},(6,):{"o":None, "c":None},},
                                ('NIFTY50',)    :{(0,):{"o":None, "c":None},(1, 2, 3, 4, 5)   :{"o":"04:00", "c":"09:59"}, (6,):{"o":None, "c":None},}
                            },
                        'europe_index_Cluster':
                            {
                                ('AEX25', 'CAC40', 'DAX', 'GER40', 'ESTX50', 'EUR50', 'FTSE', 'UK100', 'IBEX', 'SMI', 'OMXS30',):  # внутрисуточный разрыв 22:00 – 22:59 UTC
                                    {(0,):{"o":"23:00", "c":"23:59"}, (1, 2, 3, 4,):{"o":"00:00", "c":"23:59"}, (5,):{"o":"00:00", "c":"21:59"},(6,):{"o":None, "c":None},},  # # # внутрисуточный разрыв 22:00 – 22:59 UTC
                                ('SPAIN35', 'ITALY40', 'IT40',):
                                    {(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'07:00', "c":'19:59'},(6,):{"o":None, "c":None},},
                                ('SLI', 'NL25', 'BEL20',):
                                    {(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'08:15', "c":'16:30'},(6,):{"o":None, "c":None},},
                                ('CH20',)   :{(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'07:00', "c":'20:59'},(6,):{"o":None, "c":None},},
                                ('NO25',)   :{(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'08:00', "c":'15:15'},(6,):{"o":None, "c":None},},
                                ('WIG20',)  :{(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'08:00', "c":'15:45'},(6,):{"o":None, "c":None},},
                                ('HKMYA',)  :{(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'07:00', "c":'15:00'},(6,):{"o":None, "c":None},},
                                ('SAFRI40',):{(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'06:00', "c":'15:30'},(6,):{"o":None, "c":None},}
                            },
                        'america_index_Cluster':        #АМЕРИКА 13:30 – 20:00 UTC (зимой: 14:30 – 21:00)
                            {
                                ('SP500','SPCOMP','DJI','NDX','US100','RUT','US2000','US30',):
                                                {(0,):{"o":"23:00", "c":"23:59"}, (1, 2, 3, 4,):{"o":"00:00", "c":"23:59"}, (5,):{"o":"00:00", "c":"21:59"},(6,):{"o":None, "c":None},},
                                ('CA60', 'CAN60', 'ME0000', 'DJT', 'DJU','DJC','NYSEI',):
                                        {(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'14:30', "c":'21:59'},(6,):{"o":None, "c":None},},
                                ('NACOMP',):    {(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'14:45', "c":'22:15'},(6,):{"o":None, "c":None},},
                                ('VIX', 'VIX_SPOT'):                                                            # внутрисуточный разрыв 22:00 – 22:59 UTC
                                                {(0,):{"o":"23:00", "c":"23:59"},(1, 2, 3, 4,):{"o":'00:00', "c":'21:59'}, (5,):{"o":'00:00', "c":'21:59'},(6,):{"o":None, "c":None},},
                            },
                        'word_index_Cluster':                                                          # внутрисуточный разрыв 22:00 – 22:59 UTC
                            {
                                ('USDX',):{(0,):{"o":"23:00", "c":"23:59"}, (1, 2, 3, 4,):{"o":"00:00", "c":"23:59"}, (5,):{"o":"00:00", "c":"21:59"},(6,):{"o":None, "c":None},},
                            },

                            

                    }

    |
    Фактическое котирование между Американским и Европейским летним переходом по кластерам:
    'Energy_Cluster', 'Metals_Cluster', 'Commodities_Cluster'
    проверка проведена вручную.
    # https://www.notion.so/b0arding/METALS-Energy-COMMODITIES-3224f60c694580548265dea546349dac

In [ ]:
# Фактическое котирование между Американским и Европейским летним переходом по кластерам 'Energy_Cluster', 'Metals_Cluster',  'Commodities_Cluster' проверка проведена вручную.
# https://www.notion.so/b0arding/METALS-Energy-COMMODITIES-3224f60c694580548265dea546349dac
# лист в notions со спецификаией активов, Выполнено 17 марта 2026
index_sessions_dict = {
                        'Energy_Cluster':
                            {
                                ('BRTSPOT', 'Gasoil',):     # Внутрисуточный разрыв 22:00 – 01:00 UTC # котировки в ночное время в начале сессии поступают неравномерно
                                    {(0,):{"o":None, "c":None}, (1, 2, 3, 4, 5,):{"o":"00:00", "c":"20:59"}, (6,):{"o":None, "c":None},},
                                ('WTISPOT', 'Gas', 'NG',):  # Внутрисуточный разрыв 22:00 – 22:59 UTC
                                    {(0,):{"o":"22:00", "c":"23:59"}, (1, 2, 3, 4,):{"o":"00:00", "c":"23:59"}, (5,):{"o":"00:00", "c":"20:59"},  (6,):{"o":None, "c":None},},
                            },
                        'Metals_Cluster':
                            {
                                ('XAGUSD', 'XPDUSD', 'XPTUSD',):
                                    {(0,):{"o":"22:00", "c":"23:59"}, (1, 2, 3, 4,):{"o":"00:00", "c":"23:59"}, (5,):{"o":"00:00", "c":"20:59"}, (6,):{"o":None, "c":None},},
                                ('XAUUSD',):
                                    {(0,):{"o":"23:00", "c":"23:59"}, (1, 2, 3, 4,):{"o":"00:00", "c":"23:59"}, (5,):{"o":"00:00", "c":"21:59"}, (6,):{"o":None, "c":None},},
                            },
                        'Commodities_Cluster':
                            {
                                ('Copper',):
                                    {(0,):{"o":"22:00", "c":"23:59"}, (1, 2, 3, 4,):{"o":"00:00", "c":"23:59"}, (5,):{"o":"00:00", "c":"20:59"}, (6,):{"o":None, "c":None},},
                                ('Aluminium', 'Lead', 'Nickel', "Zink",):
                                    {(0,):{"o":None, "c":None}, (1, 2, 3, 4, 5,):{"o":"01:00", "c":"18:59"}, (6,):{"o":None, "c":None},},
                                ('CornX', 'RghRice', 'Soybean', 'Wheat', 'WheatX',):
                                    {(0,):{"o":None, "c":None}, (1, 2, 3, 4, 5,):{"o":"00:00", "c":"18:10"}, (6,):{"o":None, "c":None},},
                                ('Cotton',):
                                    {(0,):{"o":None, "c":None}, (1, 2, 3, 4, 5,):{"o":"01:00", "c":"18:15"}, (6,):{"o":None, "c":None},},
                                ("OJ",):
                                    {(0,):{"o":None, "c":None}, (1, 2, 3, 4, 5,):{"o":"02:00", "c":"18:00"}, (6,):{"o":None, "c":None},},
                                ('Sugar',):
                                    {(0,):{"o":None, "c":None}, (1, 2, 3, 4, 5,):{"o":"08:30", "c":"17:00"}, (6,):{"o":None, "c":None},},
                                ('SugarUK',):
                                    {(0,):{"o":None, "c":None}, (1, 2, 3, 4, 5,):{"o":"08:45", "c":"18:00"}, (6,):{"o":None, "c":None},},
                                ('Cocoa',):
                                    {(0,):{"o":None, "c":None}, (1, 2, 3, 4, 5,):{"o":"09:45", "c":"17:30"}, (6,):{"o":None, "c":None},},
                                ('Coffee', 'CoffeeX',):
                                    {(0,):{"o":None, "c":None}, (1, 2, 3, 4, 5,):{"o":"09:15", "c":"18:30"}, (6,):{"o":None, "c":None},},
                            },
                    }


In [ ]:
# Переводим товарные рынки и Европейские Индексы на летнее время <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
index_sessions_dict = {
                        'europe_index_Cluster':
                            {
                                ('SPAIN35', 'ITALY40', 'IT40',):
                                    {(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'06:00', "c":'18:59'},(6,):{"o":None, "c":None},},
                                ('SLI', 'NL25', 'BEL20',):
                                    {(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'07:15', "c":'15:30'},(6,):{"o":None, "c":None},},
                                ('CH20',)   :{(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'06:00', "c":'19:59'},(6,):{"o":None, "c":None},},
                                ('NO25',)   :{(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'07:00', "c":'14:15'},(6,):{"o":None, "c":None},},
                                ('WIG20',)  :{(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'07:00', "c":'14:45'},(6,):{"o":None, "c":None},},
                                ('HKMYA',)  :{(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'06:00', "c":'14:00'},(6,):{"o":None, "c":None},},
                                ('SAFRI40',):{(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'05:00', "c":'14:30'},(6,):{"o":None, "c":None},},
                            },
                        'Commodities_Cluster':
                            {
                                ('Sugar',)  :{(0,):{"o":None, "c":None}, (1, 2, 3, 4, 5,):{"o":"07:30", "c":"16:00"}, (6,):{"o":None, "c":None},},
                                ('SugarUK',): {(0,):{"o":None, "c":None}, (1, 2, 3, 4, 5,):{"o":"07:45", "c":"17:00"}, (6,):{"o":None, "c":None},},
                                ('Cocoa',)  :{(0,):{"o":None, "c":None}, (1, 2, 3, 4, 5,):{"o":"08:45", "c":"16:30"}, (6,):{"o":None, "c":None},},
                                ('Coffee', 'CoffeeX',):
                                            {(0,):{"o":None, "c":None}, (1, 2, 3, 4, 5,):{"o":"08:15", "c":"17:30"}, (6,):{"o":None, "c":None},},
                            },
                        }

In [ ]:
index_sessions_dict = { # АМЕРИКА Для перевода на летнее время <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

                        'america_index_Cluster':        #АМЕРИКА 13:30 – 20:00 UTC (зимой: 14:30 – 21:00)
                            {
                                ('SP500','SPCOMP','DJI','NDX','US100','RUT','US2000','US30',):
                                                {(0,):{"o":"22:00", "c":"23:59"}, (1, 2, 3, 4,):{"o":"00:00", "c":"23:59"}, (5,):{"o":"00:00", "c":"20:59"},(6,):{"o":None, "c":None},},
                                ('CA60', 'CAN60', 'ME0000', 'DJT', 'DJU','DJC','NYSEI',):
                                        {(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'13:30', "c":'20:59'},(6,):{"o":None, "c":None},},
                                ('NACOMP',):    {(0,):{"o":None, "c":None},(1, 2, 3, 4, 5,):{"o":'13:45', "c":'21:15'},(6,):{"o":None, "c":None},},
                                ('VIX', 'VIX_SPOT'):                                                            # внутрисуточный разрыв 22:00 – 22:59 UTC
                                                {(0,):{"o":"22:00", "c":"23:59"},(1, 2, 3, 4,):{"o":'00:00', "c":'20:59'}, (5,):{"o":'00:00', "c":'20:59'},(6,):{"o":None, "c":None},},
                            },

                    }

In [ ]:
index_sessions_dict = { # АМЕРИКА Для перевода на летнее время <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

                        'america_index_Cluster':        #АМЕРИКА 13:30 – 20:00 UTC (зимой: 14:30 – 21:00)
                            {
                                ('LIN.N','APD.N',):
                                                {(0,):{"o":None, "c":None}, (1, 2, 3, 4, 5,):{"o":"13:30", "c":"20:59"}, (6,):{"o":None, "c":None},},
                            },
                        'JP':
                            {
                                ('8088.JP',):
                                                {(0,):{"o":None, "c":None}, (1, 2, 3, 4, 5,):{"o":"00:00", "c":"06:30"}, (6,):{"o":None, "c":None},},
                            },

                    }

In [ ]:
#функция, которая позволяет добавить новые строки в DataFrame с указанными значениями в определённых колонках, а все остальные колонки оставить пустыми (NaN
import numpy as np

def add_rows_with_values(df, rows_list):
    """ Добавляет новые строки в DataFrame.
    
    Параметры:
        df         — исходный DataFrame
        rows_list  — список словарей, где каждый словарь описывает одну новую строку.
                     Ключи = имена колонок, значения = то, что нужно записать.
                     Колонки, которых нет в словаре, будут NaN.
    
    Пример:
        rows_list = [
            {'name': 'IBM', 'groupName': 'basic', 'markUpBid': 0.5},
            {'name': 'AAPL', 'groupName': 'silver', 'spreadDiffPercent': 0.1}
        ]
    """
    if not isinstance(rows_list, list): rows_list = [rows_list]     # если передали один словарь
    new_rows_df = pd.DataFrame(rows_list)                           # Создаём DataFrame из списка словарей
    
    for col in df.columns:                                          # Добавляем недостающие колонки (заполняем NaN)
        if col not in new_rows_df.columns:
            new_rows_df[col] = np.nan
    
    new_rows_df = new_rows_df[df.columns]                           # Приводим порядок колонок к оригинальному
    result_df = pd.concat([df, new_rows_df], ignore_index=True)     # Объединяем
    print(f"Добавлено {len(new_rows_df)} новых строк. Всего строк теперь: {len(result_df)}")
    
    return result_df

In [ ]:
rows_to_add = [
    {'name': 'LIN.N', 'symbolId':10220},
    {'name': 'APD.N', 'symbolId':10221},
    {'name': '8088.JP', 'symbolId':10222}
]

new_df = add_rows_with_values(symbols_df, rows_to_add)


In [ ]:
# [ФУНКЦИЯ] Создаёт словарь DataFrame-ов по кластерам + итоговый объединённый DataFrame на основе тикерных имен.
def create_sessions_dfs_v2(index_sessions_dict, symbols_df, symbol_col_name='name'):
    """
    Создаёт словарь DataFrame-ов по кластерам + итоговый объединённый DataFrame
    на основе тикерных имен.
    
    Параметры:
        index_sessions_dict: словарь с настройками сессий (новая структура).
        symbols_df: DataFrame с колонками ['symbolId', 'symbolName', ...].
        symbol_col_name: имя колонки в symbols_df, где хранятся строковые названия тикеров (например 'SP500').
    
    Возвращает:
        dict: {cluster_name: DataFrame сессий}
        pd.DataFrame: объединённый df_sessions_all
    """
    cluster_dfs = {}
    all_rows = []

    for cluster_name, groups_data in index_sessions_dict.items():           # 1. Проходим по каждому крупному кластеру (Asia, Europe...)
        cluster_rows = []
                                                                            # 2. Проходим по каждой группе тикеров внутри кластера
        # tickers_tuple — это ключ, например ('AUS200', 'N225')
        # schedule_rules — это значение, словарь с расписанием
        for tickers_tuple, schedule_rules in groups_data.items():
                                                                            # 3. Находим реальные symbolId для этих тикеров
            # Фильтруем symbols_df: ищем строки, где имя символа входит в наш кортеж tickers_tuple
            matched_symbols = symbols_df[symbols_df[symbol_col_name].isin(tickers_tuple)]
            
            if matched_symbols.empty:
                # Раскомментируйте, если хотите видеть предупреждения о ненайденных тикерах
                print(f"Warning: В symbols_df не найдены тикеры из группы {tickers_tuple}")
                continue
            
            # Получаем список пар (ID, Name) для найденных инструментов
            # Чтобы потом точно знать, к какому ID привязывать расписание
            found_ids = matched_symbols['symbolId'].tolist()
            
            # 4. Генерируем строки расписания для каждого найденного ID
            for sid in found_ids:
                
                # Применяем правила расписания
                # days_tuple: (1, 2, 3, 4, 5)
                # times: {"o": "00:00", "c": "06:30"}
                for days_tuple, times in schedule_rules.items():
                    for day in days_tuple:
                        row = {
                            "symbolId": sid,
                            "cluster": cluster_name,
                            "day": day,
                            "open": times["o"],
                            "close": times["c"]
                        }
                        cluster_rows.append(row)
                        all_rows.append(row)

        # 5. Создаём DataFrame для текущего кластера, если есть данные
        if cluster_rows:
            df_cluster = pd.DataFrame(cluster_rows)
            # Сортируем для красоты
            cluster_dfs[cluster_name] = df_cluster.sort_values(["symbolId", "day"]).reset_index(drop=True)
        else:
            cluster_dfs[cluster_name] = pd.DataFrame() # Пустой DF, если ничего не нашли

    # 6. Итоговый общий DataFrame
    if all_rows:
        df_sessions_all = pd.DataFrame(all_rows)
        df_sessions_all = df_sessions_all.sort_values(["symbolId", "day"]).reset_index(drop=True)
    else:
        df_sessions_all = pd.DataFrame()

    return cluster_dfs, df_sessions_all


index_clusters_df, index_df = create_sessions_dfs_v2(index_sessions_dict, new_df, symbol_col_name='name')

imported["pd_set_option"]("index_df", index_df, 50)

In [ ]:
imported["save_data_log_work_file"](index_df, "_sessions.csv", directory_data_temp_files, directory_data_log_files)

In [ ]:
# Множество по колонке symbolId
symbolId_set = set(index_df["symbolId"].dropna())
print(len(symbolId_set))
print(symbolId_set)

In [ ]:
# Словарь Кластеров Торговых сессий которые можно сформировать по marketID (т.е. внутри группы рыночных активов время одинаковое) <<<<<<<<<<<<<<<<<<<<<<
# "Japan_Cluster", "FOREX_Cluster", "Crypto_Cluster", "Stocks_USA_Cluster", "Stocks_Europe_Cluster"
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''' 
similar_clusters_trading_sessions = {# однотипные кластеры торговых сессий                 
                "Japan_Cluster":   # Отсутствует Переход на летнее время
                    {"markets_name": {'CFDs - Stocks Japan':18},
                    "sessions":{(0, 6):{"o":None, "c":None}, (1, 2, 3, 4, 5):{"o":"00:00", "c":"06:30"}}},  
                "FOREX_Cluster":
                    {"markets_name": {'Minor':2, 'Major':3, 'Exotic':77},
                    "sessions":{(0,):{"o":"22:00", "c":"23:59"},(1, 2, 3, 4):{"o":"00:00", "c":"23:59"}, (5,):{"o":"00:00", "c":"21:59"}, (6,):{"o":None, "c":None}}},
                "Crypto_Cluster":
                    {"markets_name": {'CRYPTO':69, 'Crypto':70, 'Crypto - USDT':71, 'Crypto STD':72},
                    "sessions":{(0, 1, 2, 3, 4, 5, 6):{"o":"00:00", "c":"23:59"}}},
                "Stocks_USA_Cluster":
                    {"markets_name": {'CFDs - Stocks United States': 33, 'ETFs': 76, 'CFDs - Stocks Canada': 22, 'CFDS - Stocks Brazil': 17},
                    "sessions":{(0, 6):{"o":None, "c":None}, (1, 2, 3, 4, 5):{"o":"14:30", "c":"20:59"}}},
                "Stocks_Europe_Cluster":
                    {"markets_name": {'CFDs - Stocks Germany': 19,      'CFDs - Stocks France': 20, 'CFDs - Stocks Italy': 34,      'CFDs - Stocks United Kingdom': 28,
                                      'CFDs - Stocks Netherlands': 13,  'CFDs - Stocks Spain': 35,  'CFDs - Stocks Switzerland': 9, 'CFDs - Stocks Sweden': 21,
                                      'CFDs - Stocks Belgium': 16,      'CFDs - Stocks Austria': 15,'CFDs - Stocks Portugal': 32,   'CFDs - Stocks Finland': 25},
                    "sessions":{(0, 6):{"o":None, "c":None}, (1, 2, 3, 4, 5):{"o":"08:00", "c":"16:30"}}}, # Зимнее время
                }
# Не используется
#exception_ssessions_dict ={"ETF__Cluster": {"markets_name": {'ETFs': 76}, "sessions":{(0, 6):{"o":None, "c":None}, (1, 2, 3, 4, 5):{"o":"07:00", "c":"15:30"}}}}

In [ ]:
# Словарь для перехода на летнее время <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
similar_clusters_trading_sessions = {# однотипные кластеры торговых сессий 
                "FOREX_Cluster":
                    {"markets_name": {'Minor':2, 'Major':3, 'Exotic':77},
                    "sessions":{(0,):{"o":"21:00", "c":"23:59"},(1, 2, 3, 4):{"o":"00:00", "c":"23:59"}, (5,):{"o":"00:00", "c":"20:59"}, (6,):{"o":None, "c":None}}},
                "Stocks_Europe_Cluster":
                    {"markets_name": {'CFDs - Stocks Germany': 19,      'CFDs - Stocks France': 20, 'CFDs - Stocks Italy': 34,      'CFDs - Stocks United Kingdom': 28,
                                      'CFDs - Stocks Netherlands': 13,  'CFDs - Stocks Spain': 35,  'CFDs - Stocks Switzerland': 9, 'CFDs - Stocks Sweden': 21,
                                      'CFDs - Stocks Belgium': 16,      'CFDs - Stocks Austria': 15,'CFDs - Stocks Portugal': 32,   'CFDs - Stocks Finland': 25},
                    "sessions":{(0, 6):{"o":None, "c":None}, (1, 2, 3, 4, 5):{"o":"07:00", "c":"15:30"}}}, # Летнее Время
                                    }

In [ ]:
sessions_dict = {                                                                                       # Stocks_USA_Cluster Летнее время
                "Stocks_USA_Cluster":
                    {"markets_name": {'CFDs - Stocks United States': 33, 'ETFs': 76, 'CFDs - Stocks Canada': 22, 'CFDS - Stocks Brazil': 17},
                    "sessions":{(0, 6):{"o":None, "c":None}, (1, 2, 3, 4, 5):{"o":"13:30", "c":"19:59"}}},
                }

In [ ]:
# [ФУНКЦИЯ] Для создания расписания из Кластеров с единым расписанием внутри. Создаёт словарь DataFrame-ов по кластерам + итоговый объединённый DataFrame
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
def create_sessions_dfs(sessions_dict, symbols_df):
    """
    Создаёт словарь DataFrame-ов по кластерам + итоговый объединённый DataFrame.
    
    Параметры:
        sessions_dict: словарь кластеров
        symbols_df: DataFrame с колонками ['symbolId', 'marketId']
    
    Возвращает:
        dict: {cluster_name: DataFrame сессий}
        pd.DataFrame: объединённый df_sessions_all
    """
    cluster_dfs = {}
    all_rows = []

    for cluster_name, cluster_data in sessions_dict.items():
        # Получаем все marketId кластера
        market_ids = list(cluster_data["markets_name"].values())
        
        # Фильтруем symbolId, относящиеся к этому кластеру
        symbol_ids = symbols_df[
            symbols_df["marketId"].isin(market_ids)
        ]["symbolId"].tolist()
        
        if not symbol_ids:
            print(f"Кластер '{cluster_name}' — нет символов")
            continue
        
        # Формируем строки только для этого кластера
        rows = []
        for sid in symbol_ids:
            for days_tuple, times in cluster_data["sessions"].items():
                for day in days_tuple:
                    rows.append({
                        "symbolId": sid,
                        "cluster": cluster_name,  # ← добавляем имя кластера
                        "day": day,
                        "open": times["o"],
                        "close": times["c"]
                    })
        
        # Создаём DataFrame для кластера
        cluster_df = pd.DataFrame(rows)
        cluster_dfs[cluster_name] = cluster_df.sort_values(["symbolId", "day"]).reset_index(drop=True)
        
        # Добавляем строки в общий список
        all_rows.extend(rows)

    # Итоговый DataFrame — все кластеры вместе
    df_sessions_all = pd.DataFrame(all_rows)
    df_sessions_all = df_sessions_all.sort_values(["symbolId", "day"]).reset_index(drop=True)

    return cluster_dfs, df_sessions_all
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

sessions_dict = similar_clusters_trading_sessions # присваиваем переменной словарь по которому будем создавать расписание
# Запуск функции. cluster_dfs - словарь DataFrame-ов по кластерам, df_sessions_all - итоговый DataFrame
cluster_dfs, df_sessions_all = create_sessions_dfs(sessions_dict, symbols_df)

print("cluster_dfs keys: ", cluster_dfs.keys())

In [ ]:
imported["pd_set_option"]("df_sessions_all", df_sessions_all, 10)
# Множество по колонке symbolId
symbolId_set = set(df_sessions_all["symbolId"].dropna())
print(len(symbolId_set))
print(symbolId_set)

In [ ]:

# Сохраняем в CSV Новое расписание по методу обработки отдельно каждого символа <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
df_sessions_new = process_sessions_conditions( # вызываем [ФУНКЦИЮ]
    table_name = 'index_df', dfs = index_df, symbols_df = symbols_df, directory_data_temp_files = directory_data_temp_files, directory_data_log_files=directory_data_log_files)

In [ ]:
# Если нужно использовать расписание сессий уже существующее и обновить его, после чего загрузить на сервер <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# original_sessions
# пути к файлам с данными для подключения к SQL серверу <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
dot_big_sql_main_cred= "own_platform\\credits\\own_platforn_sql_main_main_01.txt"           # Получаем Данные с ПРОД сервера
#dot_big_sql_main_cred= "own_platform\\credits\\own_platforn_sql_main_stage_01.txt"

original_main_symbols_sessions_df = imported["get_sql_tab"]   ("SELECT * FROM `symbolsSessions`",    dot_big_sql_main_cred)
#imported["pd_set_option"]("original_main_symbols_sessions_df", original_main_symbols_sessions_df, 50)

# df_sessions_new взять множество по колонке symbolId;
# original_main_symbols_sessions_df удалить все строки в которых значение по колонке symbolId входят в множество;
# добавить строки df_sessions_new в original_main_symbols_sessions_df, при этом проверив что в в df_sessions_new есть необходимые колонки,
# данные из колонок  df_sessions_new которых нет в original_main_symbols_sessions_df не добавлять.
new_ids = set(df_sessions_new['symbolId'])
common_cols = df_sessions_new.columns.intersection(original_main_symbols_sessions_df.columns)
update_main_symbols_sessions_df = pd.concat([
    original_main_symbols_sessions_df[~original_main_symbols_sessions_df['symbolId'].isin(new_ids)],
    df_sessions_new[common_cols]
], ignore_index=True)
imported["save_data_log_work_file"](update_main_symbols_sessions_df, "update_main_symbols_sessions_df.csv", directory_data_temp_files, directory_data_log_files)
imported["pd_set_option"]("update_main_symbols_sessions_df", update_main_symbols_sessions_df, 5)

In [ ]:
"""Передаём ДФ и список колонок.
Функия проверяет нет ли строк дубликатов в которых данные одинаковые по этим колонкам.
Если такие строки есть то формируется ДФ с данными по этим колонкам, дополнительной колонкой
в которой указанно сколько раз встречается и дополнитеьной колонкой в которой указанны индексы этих строк"""
def find_duplicates(df, subset_cols):
    """Короткая версия"""
    dup = df[df.duplicated(subset=subset_cols, keep=False)].copy()
    if dup.empty:
        print("Дубликатов не найдено.")
        return pd.DataFrame()
    
    dup['count'] = dup.groupby(subset_cols).transform('size')
    dup['indices'] = dup.index
    return dup.sort_values(by=subset_cols + ['count'])

find_duplicates(update_main_symbols_sessions_df, ['symbolId',	'type',	'day'])

In [ ]:
imported["pd_set_option"]("index_df", index_df, 50)

In [ ]:
# 1. Проверяем идентичность названий и порядка колонок
if list(df_sessions_all.columns) == list(index_df.columns):
    # 2. Соединяем построчно
    result = pd.concat([df_sessions_all, index_df], ignore_index=True)
    print("Датафреймы успешно соединены.")
    process_sessions_conditions(table_name='total_sessions',dfs=result,symbols_df=symbols_df,directory_data_temp_files=directory_data_temp_files,directory_data_log_files=directory_data_log_files)
else:
    print("Ошибка: Колонки не идентичны или нарушен их порядок!")
    # Поиск различий (опционально)
    diff = set(df_sessions_all.columns) ^ set(index_df.columns)
    if diff:
        print(f"Разные колонки: {diff}")
    else:
        print("Названия совпадают, но порядок разный.")

In [ ]:

cluster_groups = {
                    'sessions': {
                                'df': df_sessions_all,
                                'clusters': ['Japan_Cluster', 'FOREX_Cluster', 'Crypto_Cluster', 'Stocks_USA_Cluster', 'Stocks_Europe_Cluster', 'Metals_Cluster']
                                },
                    'indices': {
                                'df': index_df,
                                'clusters': ['asia_oceania_index_Cluster', 'europe_index_Cluster', 'america_index_Cluster', 'word_index_Cluster']
                                }
                }
for group_name, info in cluster_groups.items():
    source_df = info['df']
    for cluster_name in info['clusters']:
        # Most common case: cluster_name is a column in the DataFrame
        if cluster_name in source_df.columns:
            df_cluster = source_df[cluster_name]
        else:
            df_cluster = source_df  # or handle differently
            
        process_sessions_conditions(table_name=cluster_name,dfs=df_cluster,symbols_df=symbols_df,directory_data_temp_files=directory_data_temp_files,directory_data_log_files=directory_data_log_files)

In [ ]:
#index_clusters_df, index_df

table_name = 'asia_oceania_index_Cluster'
df = index_clusters_df[table_name]
process_sessions_conditions(table_name, df, symbols_df, directory_data_temp_files, directory_data_log_files)

table_name = 'europe_index_Cluster'
process_sessions_conditions(table_name, df, symbols_df, directory_data_temp_files, directory_data_log_files)

table_name = 'america_index_Cluster'
process_sessions_conditions(table_name, df, symbols_df, directory_data_temp_files, directory_data_log_files)

table_name = 'word_index_Cluster'
process_sessions_conditions(table_name, df, symbols_df, directory_data_temp_files, directory_data_log_files)

In [ ]:
# Формируем конфиги по кластеру 'Stocks_Europe_Cluster':
table_name = 'Stocks_Europe_Cluster'
df = cluster_dfs[table_name]
process_sessions_conditions(table_name, df, symbols_df, directory_data_temp_files, directory_data_log_files)

# Формируем конфиги по кластеру 'Stocks_Europe_Cluster':
table_name = 'Stocks_USA_Cluster'
df = cluster_dfs[table_name]
process_sessions_conditions(table_name, df, symbols_df, directory_data_temp_files, directory_data_log_files)

In [ ]:
print("cluster_dfs keys: ", cluster_dfs.keys())

imported["pd_set_option"]("df_sessions_all", cluster_dfs["FOREX Cluster"], 50)


imported["pd_set_option"]("df_sessions_all", df_sessions_all, 50)


In [ ]:
# Список для накопления строк
rows = []

# Проходим по всем кластерам
for cluster_name, cluster_data in sessions_dict.items():
    # Собираем все marketId из этого кластера
    market_ids = tuple(cluster_data["markets_name"].values())  # например (18,) или (2, 3, 77)
    
    # Фильтруем symbolId по marketId
    symbol_ids = symbols_df[
        symbols_df["marketId"].isin(market_ids)
    ]["symbolId"].tolist()
    
    if not symbol_ids:
        print(f"Предупреждение: для кластера '{cluster_name}' не найдено symbolId")
        continue
    
    # Для каждого symbolId добавляем все дни из сессий кластера
    for sid in symbol_ids:
        for days_tuple, times in cluster_data["sessions"].items():
            for day in days_tuple:
                rows.append({
                    "symbolId": sid,
                    "day": day,
                    "open": times["o"],
                    "close": times["c"]
                })

# Создаём итоговый DataFrame
df_sessions = pd.DataFrame(rows)

# Сортируем по symbolId и дню недели
df_sessions = df_sessions.sort_values(["symbolId", "day"]).reset_index(drop=True)

# Результат
imported["pd_set_option"]("df_sessions", df_sessions, 50)

# Опционально: проверить количество строк
print(f"\nВсего строк: {len(df_sessions)}")
print(f"Уникальных symbolId: {df_sessions['symbolId'].nunique()}")

In [ ]:
imported["pd_set_option"]("df_sessions_transposed", df_sessions_transposed, 5)

In [ ]:
sessions_dict = {
    "Japan Cluster": {
        "markets_name": {'CFDs - Stocks Japan': 18},
        "sessions": {(0, 6): {"o": None, "c": None}, (1, 2, 3, 4, 5): {"o": "00:00", "c": "06:00"}
        }
    }
}

In [ ]:
import pandas as pd

sessions_dict = {
    "Japan Cluster":{
        "markets_name": {'CFDs - Stocks Japan':18},
        "sessions": {(0, 6): {"o": None, "c": None}, (1, 2, 3, 4, 5): {"o": "00:00", "c": "06:00"}
        }
    },  
}

# Берём кластер
cluster = sessions_dict["Japan Cluster"]
market_ids = tuple(cluster["markets_name"].values())  # (18,)

# Фильтруем symbolId по marketId
symbol_ids = symbols_df[symbols_df["marketId"].isin(market_ids)]["symbolId"].tolist()
print("symbol_ids: ", symbol_ids)

# Формируем строки для DataFrame
rows = [
    {"symbolId": sid, "day": day, "open": val["o"], "close": val["c"]}
    for sid in symbol_ids
    for days_tuple, val in cluster["sessions"].items()
    for day in days_tuple
]

# Создаём DataFrame
df_sessions = pd.DataFrame(rows)

# Сортируем по symbolId и day
df_sessions = df_sessions.sort_values(["symbolId", "day"]).reset_index(drop=True)

print(df_sessions)


In [ ]:
# Функция для конвертации времени в минуты
def time_to_minutes(t):
    if t is None:
        return None
    hours, minutes = map(int, t.split(":"))
    return hours * 60 + minutes

# Применяем ко всем колонкам open и close
for col in ["open", "close"]:
    df_sessions_transposed[col] = df_sessions_transposed[col].apply(time_to_minutes)

print(df_sessions_transposed)

In [ ]:
df_sessions_transposed['type'] = 1
import pandas as pd

# Исходный DataFrame
# df_sessions_transposed уже существует и в нём есть колонка 'type' = 1

# 1. Создаём копию для дубликатов
df_duplicates = df_sessions_transposed.copy()
df_duplicates['type'] = 0  # меняем type на 0

# 2. Конкатенируем оригинал и дубликаты
df_combined = pd.concat([df_sessions_transposed, df_duplicates], ignore_index=True)

# 3. Сортируем так, чтобы дубликат шёл сразу после оригинала
# создаём вспомогательный индекс: оригинал 0, дубликат 1
df_combined['_dup'] = [0,1]* (len(df_sessions_transposed))
df_combined = df_combined.sort_values(['symbolId','day','_dup']).drop(columns='_dup').reset_index(drop=True)

# 4. Обновляем исходный df
df_sessions_transposed = df_combined

print(df_sessions_transposed)


In [ ]:
imported["save_data_log_work_file"](df_sessions_transposed, "df_sessions_JP.csv", directory_data_temp_files, directory_data_log_files)

In [ ]:

# создаём список значений ключей третьего уровня соответствующих ключу второго уровня "markets_name"
# Создаём новый словарь: ключ = tuple значений markets_name, значение = sessions
result = {}
for cluster_name, cluster_data in sessions_dict.items():
    # Получаем значения markets_name
    key = tuple(cluster_data.get("markets_name", {}).values())
    # Получаем sessions
    value = cluster_data.get("sessions", {})
    # Добавляем в словарь
    result[key] = value

print(result)

In [ ]:
# Однострочный вариант
result = {
    tuple(cluster["markets_name"].values()): cluster["sessions"]
    for cluster in sessions_dict.values()
}

print(result)

In [ ]:
# Формируем ДФ
# ['symbolId'] = 18;
#  ['day'] = [0,1,2,3,4,5,6];
# ['open'] = соответствующие значения из кортеджа пречисления в котором являются днями 
# ['close'] = соответствующие значения из кортеджа пречисления в котором являются днями 

import pandas as pd

# Берём конкретный кластер
cluster = sessions_dict["Japan Cluster"]
symbol_id = list(cluster["markets_name"].values())[0]

# Создаём список словарей для DataFrame
rows = [
    {"symbolId": symbol_id, "day": day, "open": val["o"], "close": val["c"]}
    for days_tuple, val in cluster["sessions"].items()
    for day in days_tuple
]

# Создаём DataFrame
df_sessions_transposed = pd.DataFrame(rows)

# Сортируем по дню недели
df_sessions_transposed = df_sessions_transposed.sort_values("day").reset_index(drop=True)

print(df_sessions_transposed)


In [ ]:
df_sessions_enriched_sorted = df_sessions_enriched.sort_values(
    by="trade_open_0",      # колонка для сортировки
    ascending=False,        # True — по возрастанию, False — по убыванию
    ignore_index=True       # сброс индексов после сортировки
)
df_filtered = df_sessions_enriched[df_sessions_enriched["name_s"] == "EURUSD"]
imported["pd_set_option"]("df_filtered", df_filtered, 5)


imported["pd_set_option"]("df_sessions_enriched_sorted", df_sessions_enriched_sorted, 5)

In [ ]:
imported["save_data_log_work_file"](df_sessions_enriched, "df_sessions.csv", directory_data_temp_files, directory_data_log_files)

In [ ]:
# df_sessions_filtered создать следующим образом:
#  взять колонки symbolId, name, displayName из  df_filtered
